In [28]:
import torch
import torch.nn as nn

In [29]:
class GroupedLayerNorm(nn.Module):
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        if num_channels % num_groups != 0:
            raise ValueError("num_channels must be divisible by num_groups")
        self.gn = nn.GroupNorm(num_groups, num_channels, eps=eps, affine=affine)
        self.num_channels = num_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        original_shape = x.shape
        x = x.reshape(-1, self.num_channels)
        x = self.gn(x)
        x = x.view(original_shape)
        return x
    
        
class PermutedGroupNorm(nn.Module):
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.gn = nn.GroupNorm(num_groups, num_channels, eps=eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.movedim(-1, 1)
        x = self.gn(x)
        x = x.movedim(1, -1)
        return x


In [30]:
def check_equivalence():
    # 1. 設定
    N, H, W, C = 2, 16, 16, 32  # バッチサイズ2, 16x16画像, 32チャネル
    num_groups = 4
    
    # 2. 入力データの作成 (N, H, W, C)
    # これは Permute([0, 2, 3, 1]) 済みのデータを想定しています
    torch.manual_seed(42)
    x = torch.randn(N, H, W, C)
    
    # 3. モデルのインスタンス化
    model_A = GroupedLayerNorm(num_groups, C, affine=True)
    model_B = PermutedGroupNorm(num_groups, C)
    
    # 4. 重み(weight)とバイアス(bias)の同期
    # 厳密な比較のため、初期値を完全に一致させます
    with torch.no_grad():
        model_B.gn.weight.copy_(model_A.gn.weight)
        model_B.gn.bias.copy_(model_A.gn.bias)
    
    # 5. 推論実行
    out_A = model_A(x)
    out_B = model_B(x)
    
    # 6. 差分の確認
    diff = (out_A - out_B).abs().max().item()
    is_equivalent = torch.allclose(out_A, out_B, atol=1e-6)
    
    print(f"Max difference: {diff:.6f}")
    print(f"Is equivalent (atol=1e-6)?: {is_equivalent}")

    # 挙動の違いを確認するための統計量チェック
    # GroupedLayerNormは「ピクセルごと」に正規化されているか？
    # -> reshape(-1, C) 後の各行(各ピクセル)のグループごとの分散を確認
    reshaped_A = out_A.reshape(-1, C)
    # グループ0 (最初の C/num_groups チャネル) の最初のピクセルの標準偏差
    channels_per_group = C // num_groups
    std_pixel_0_group_0_A = reshaped_A[0, :channels_per_group].std(unbiased=False)
    
    # PermutedGroupNormは「画像全体(H,W)」で正規化されているか？
    # -> (N, C, H, W) 空間でのグループごとの標準偏差
    # out_B は (N, H, W, C) なので、チャネル次元を合わせて計算
    group_0_B = out_B[0, :, :, :channels_per_group] # (H, W, C_group)
    std_image_0_group_0_B = group_0_B.std(unbiased=False)

    print("-" * 30)
    print(f"Analysis A (Pixel-wise std target ~1.0): {std_pixel_0_group_0_A:.4f}")
    print(f"Analysis B (Spatial std target ~1.0):    {std_image_0_group_0_B:.4f}")

if __name__ == "__main__":
    check_equivalence()

Max difference: 2.036675
Is equivalent (atol=1e-6)?: False
------------------------------
Analysis A (Pixel-wise std target ~1.0): 1.0000
Analysis B (Spatial std target ~1.0):    1.0000


In [32]:
import torch
import torch.nn as nn

# --- 1. Pixel-wise GroupNorm (前回定義: Model A) ---
class GroupedLayerNorm(nn.Module):
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.gn = nn.GroupNorm(num_groups, num_channels, eps=eps, affine=affine)
        self.num_channels = num_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        original_shape = x.shape
        # (N, H, W, C) -> (N*H*W, C) : ピクセルをバッチ次元に畳み込む
        x = x.reshape(-1, self.num_channels)
        x = self.gn(x)
        x = x.view(original_shape)
        return x

# --- 2. Spatial GroupNorm (前回定義: Model B) ---
class PermutedGroupNorm(nn.Module):
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.gn = nn.GroupNorm(num_groups, num_channels, eps=eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (N, H, W, C) -> (N, C, H, W) : 空間次元を含めて正規化する標準的なGN
        x = x.movedim(-1, 1)
        x = self.gn(x)
        x = x.movedim(1, -1)
        return x

# --- 3. Split + LayerNorm (今回追加: Model C) ---
class SplitLayerNorm(nn.Module):
    def __init__(self, num_groups: int, num_channels: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        if num_channels % num_groups != 0:
            raise ValueError("num_channels must be divisible by num_groups")
        
        self.num_groups = num_groups
        self.channels_per_group = num_channels // num_groups
        
        # 各グループごとに独立したLayerNormを用意
        # elementwise_affine=affine により、学習パラメータ(weight/bias)を持つか制御
        self.layers = nn.ModuleList([
            nn.LayerNorm(self.channels_per_group, eps=eps, elementwise_affine=affine)
            for _ in range(num_groups)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, H, W, C)
        # チャネル方向(dim=-1)で分割
        chunks = x.split(self.channels_per_group, dim=-1)
        
        outputs = []
        for chunk, layer in zip(chunks, self.layers):
            # chunk: (N, H, W, C/G)
            # LayerNormは最後の次元(C/G)に対して正規化を行う
            outputs.append(layer(chunk))
            
        # 再結合
        return torch.cat(outputs, dim=-1)

# --- 検証実行 ---
def compare_three_variants():
    # 設定
    N, H, W, C = 2, 16, 16, 32
    num_groups = 4
    
    torch.manual_seed(42)
    x = torch.randn(N, H, W, C) # (N, H, W, C) 入力
    
    # モデル定義
    model_A = GroupedLayerNorm(num_groups, C)
    model_B = PermutedGroupNorm(num_groups, C)
    model_C = SplitLayerNorm(num_groups, C)
    
    # 重みの同期 (検証の肝)
    # Model A, Bは nn.GroupNorm なので weight は (C,)
    # Model C は nn.LayerNorm のリストなので、それぞれ (C/G,)
    # Aの重みを分割してCにコピーする
    with torch.no_grad():
        # A -> B (前回同様)
        model_B.gn.weight.copy_(model_A.gn.weight)
        model_B.gn.bias.copy_(model_A.gn.bias)
        
        # A -> C (分割コピー)
        chunks_w = model_A.gn.weight.split(C // num_groups)
        chunks_b = model_A.gn.bias.split(C // num_groups)
        for i, layer in enumerate(model_C.layers):
            layer.weight.copy_(chunks_w[i])
            layer.bias.copy_(chunks_b[i])

    # 推論
    out_A = model_A(x)
    out_B = model_B(x)
    out_C = model_C(x)
    
    # 比較
    diff_A_C = (out_A - out_C).abs().max().item()
    diff_B_C = (out_B - out_C).abs().max().item()
    
    print(f"Diff A (GroupedLN) vs C (SplitLN):  {diff_A_C:.8f}")
    print(f"Diff B (PermutedGN) vs C (SplitLN): {diff_B_C:.8f}")
    
    if diff_A_C < 1e-6:
        print("\n>> 結論: GroupedLayerNorm は SplitLayerNorm と等価です。")
    else:
        print("\n>> 結論: 等価ではありません。実装を確認してください。")

if __name__ == "__main__":
    compare_three_variants()

Diff A (GroupedLN) vs C (SplitLN):  0.00000048
Diff B (PermutedGN) vs C (SplitLN): 2.03667498

>> 結論: GroupedLayerNorm は SplitLayerNorm と等価です。
